#### Pulling data from kaggle

In [ ]:
import kagglehub

path = kagglehub.dataset_download("abdallahaboelkhair/heartbeat-sound")

print("Path to dataset files:", path)

In [ ]:
import os

if os.path.exists(path):
    print(f"Directory found: {path}")
     
    contents = os.listdir(path)
    print(f" Items in dataset: {contents}")
    
    # Check for audio files 
    audio_files = [f for f in contents if f.endswith('.wav') or f.endswith('.mp3')]
    if audio_files:
        print(f"Found {len(audio_files)} audio files. First 3: {audio_files[:3]}")
    else:
        # Check for subfolders 
        subfolders = [f for f in contents if os.path.isdir(os.path.join(path, f))]
        print(f"Found subfolders: {subfolders}")
else:
    print("Path not found. Download failed.")

# **Heartbeat sounds**

Specific cardiac actions, like valve closure or chordae tendineae tension, create heart sounds.
<br>S1 occurs when the mitral and tricuspid valves close.
<br>S2 occurs when the aortic and pulmonic valves close.

In medicine we call the ‘lub’
sound "S1" and the ‘dub’ sound
"S2".

Short intro about heart sounds:

https://www.youtube.com/watch?v=dBwr2GZCmQM

# About data:

The 2012 Classifying Heart Sounds Challenge, presented at AISTATS, focuses on categorizing cardiac audio. The dataset incorporates two primary sources:<br>
Source A: Audio collected from the general public using the iStethoscope Pro. <br>
Source B: Clinical trial data recorded in hospitals via the DigiScope digital stethoscope.<br>

To streamline the workflow in this notebook, we first pre-process the directory structures and concatenate both datasets (A and B) into a unified format. The original data is hosted at peterjbentley.com/heartchallenge/.

# Import main libraries

In [ ]:
import glob
import fnmatch
import pandas as pd  # loading data
import numpy as np   # applying audio logics
import librosa # To deal with Audio files
import librosa.display
import seaborn as sns
import matplotlib.pyplot as plt  # visuls
import IPython.display as ipd
import math     # math logics

# deep learning libs
import tensorflow as tf 
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import concatenate  # for joining audio

from tensorflow.keras.models import Sequential, Model, load_model

from tensorflow.keras.layers import Conv1D, Conv2D, SeparableConv1D, MaxPooling1D, MaxPooling2D
from tensorflow.keras.layers import Input, add, Flatten, Dense, BatchNormalization, Dropout, LSTM, GRU
from tensorflow.keras.layers import GlobalMaxPooling1D, GlobalMaxPooling2D, Activation, LeakyReLU, ReLU

from tensorflow.keras import regularizers
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping


from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,matthews_corrcoef
from sklearn.metrics import cohen_kappa_score,roc_auc_score,confusion_matrix,classification_report

loading

In [ ]:
data_path = os.path.join(path, "Heartbeat_Sound") 
print(os.listdir(data_path))

In [ ]:
train_data      = data_path
unlabel_data    = data_path  + "/unlabel/"

normal_data     = train_data + '/normal/'
murmur_data     = train_data + '/murmur/'
extrastole_data = train_data + '/extrastole/'
artifact_data   = train_data + '/artifact/'
extrahls_data   = train_data + "/extrahls/"

In [ ]:
print("Normal files:", len(os.listdir(normal_data))) #length of normal training sounds
print("Murmur files:",len(os.listdir(murmur_data))) #length of murmur training sounds
print("Extrastole files:", len(os.listdir(extrastole_data))) #length of extrastole training sounds
print("Artifact files:",len(os.listdir(artifact_data))) #length of artifact training sounds
print("Extrahls files:",len(os.listdir(extrahls_data))) #length of extrahls training sounds

print('TOTAL TRAIN SOUNDS:', len(os.listdir(normal_data))
                              + len(os.listdir(murmur_data))
                              + len(os.listdir(extrastole_data))
                              + len(os.listdir(artifact_data))
                              + len(os.listdir(extrahls_data)))

In [ ]:
print("Test sounds: ", len(os.listdir(unlabel_data)))

# EDA and Visualization

In [ ]:
x = np.array([len(os.listdir(normal_data)),
              len(os.listdir(murmur_data)),
              len(os.listdir(extrastole_data)),
              len(os.listdir(artifact_data)),
              len(os.listdir(extrahls_data))])
labels = ['normal', 'murmur', 'extrastole', 'artifact', 'extrahls']
plt.pie(x, labels = labels, autopct = '%.0f%%', radius= 1.5, textprops={'fontsize': 16})
plt.show()

##### The figure shows imbalanced data.


### Pick a random sound

In [ ]:
# Listen to rondom audio from specific class
def random_sound (audio_class):
    random_sound = np.random.randint(0,len(os.listdir(audio_class)))
    sound = os.listdir(audio_class)[random_sound]
    sound = audio_class+sound
    sound,sample_rate = librosa.load(sound)
    return ipd.Audio(sound,rate=sample_rate),sound

### Waveform

Sound travels as air pressure variations that reach the ear. A digital audio file is created when a sound sensor detects these mechanical waves and converts them into electrical signals. This digital representation captures the displacement of the wave over time. On a standard waveform graph:<br>
The X-axis represents the passage of time.<br>
The Y-axis measures the amplitude, which reflects the distance air molecules move from their equilibrium position.

In [ ]:
# show waveform of audio from dataset
# Amplitude is plotted along the Y-axis, representing the maximum displacement of air molecules from their resting state.
def show_audio_waveform(audio_sample):
    plt.figure(figsize=(20,5))
    librosa.display.waveshow(audio_sample, sr=22050)
    plt.title("Sound")
    plt.xlabel("Time")
    plt.ylabel("Amplitude")
    plt.show()

### Spectrum

A sound spectrum represents a sound sample by displaying the magnitude of vibration across various frequencies. Typically shown as a graph, it plots power or pressure (measured in decibels) against frequency (measured in Hertz or kilohertz).<br>

By analyzing a sound’s frequency composition, the spectrum maps these components onto a coordinate plane:<br>

X-axis: Displays the frequency (\(f\))<br>
Y-axis (Ordinate): Displays the amplitude (\(A\)) or intensity of each harmonic component.

In [ ]:
# show spectrum of audio from dataset
def show_audio_spectrum(audio_sample):
    sample_rate = 22050
    fft_normal = np.fft.fft(audio_sample)
    magnitude_normal = np.abs(fft_normal)
    freq_normal = np.linspace(0,sample_rate, len(magnitude_normal))
    half_freq = freq_normal[:int(len(freq_normal)/2)]
    half_magnitude = magnitude_normal[:int(len(freq_normal)/2)]

    plt.figure(figsize=(12,8))
    plt.plot(half_freq,half_magnitude)
    plt.title("Spectrum")
    plt.xlabel("Frequency")
    plt.ylabel("Magnitude")
    plt.show()

### Spectogram

Human hearing relies on both intensity and pitch. While intensity describes a sound's power at a specific moment, pitch refers to its frequency—where higher pitches correspond to higher frequencies. To better mirror how our brains process audio, we can use a Spectrogram, which adds the frequency dimension to our representation.

A spectrogram is a visual way to represent how a signal's frequency spectrum changes over time. When used for audio, they are often referred to as sonographs, voiceprints, or voicegrams.
<br>This tool is vital across various disciplines, including music, linguistics, sonar, speech processing, and seismology. In audio analysis, spectrograms allow for the phonetic identification of spoken words and the study of animal vocalizations. They are typically generated using methods such as Fourier transforms, wavelet transforms, optical spectrometers, or banks of band-pass filters.

In [ ]:
# show specrogram of audio from dataset
# the output is an image that represents a sound.
# X-axis is for time, y-axis is for frequency and the color is for intensity
def show_spectrogram (audio_sample):
    # STFT -> spectrogram
    hop_length = 512 # in num. of samples
    n_fft = 2048 # window in num. of samples
    sample_rate = 22050

    # calculate duration hop length and window in seconds
    hop_length_duration = float(hop_length)/sample_rate
    n_fft_duration = float(n_fft)/sample_rate

    print("STFT hop length duration is: {}s".format(hop_length_duration))
    print("STFT window duration is: {}s".format(n_fft_duration))

    # perform stft
    stft_normal = librosa.stft(audio_sample, n_fft=n_fft, hop_length=hop_length)

    # calculate abs values on complex numbers to get magnitude
    spectrogram = np.abs(stft_normal)
    log_spectrogram = librosa.amplitude_to_db(spectrogram)

    # display spectrogram
    plt.figure(figsize=(15,10))
    librosa.display.specshow(log_spectrogram, sr=sample_rate, hop_length=hop_length)
    plt.xlabel("Time")
    plt.ylabel("Frequency")
    plt.colorbar()
    #plt.set_cmap("YlOrBr")
    plt.title("Spectrogram")

### MFCCs

Raw audio signals are often too noisy for direct model input. Research indicates that performance improves significantly when models utilize extracted features rather than raw waveforms. MFCC is the industry-standard technique for this purpose.An MFC represents a sound's short-term power spectrum by applying a cosine transform to a log power spectrum on the non-linear Mel scale. Unlike standard spectra, the Mel scale uses frequency bands that mimic the human auditory system's response, making it highly effective for tasks like audio compression and speech recognition.

**Standard MFCC Extraction Pipeline:**

1.Fourier Transform: Analyze a windowed segment of the signal.<br>
2.Mel Mapping: Map the resulting power spectrum onto the Mel scale using overlapping windows.<br>
3.Logarithm: Calculate the log-powers for each Mel frequency.<br>
4.Discrete Cosine Transform (DCT): Treat the log-powers as a new signal and apply a DCT.<br>
5.Final Output: The resulting amplitudes are the MFCCs.

In [ ]:
# MFCCs
# extract 52 MFCCs
def show_mfcc_features(audio_sample):
    hop_length = 512
    n_fft = 2048
    sample_rate = 22050

    MFCCs = librosa.feature.mfcc(
        y=audio_sample,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mfcc=52   # 52 features
    )

    plt.figure(figsize=(15,10))
    librosa.display.specshow(MFCCs, sr=sample_rate, hop_length=hop_length)
    plt.xlabel("Time")
    plt.ylabel("MFCC coefficients")
    plt.colorbar()
    plt.title("MFCCs")
    plt.show()


##  Dataset Classes

### *1. Normal*

The standard resting heart rate for most individuals ranges from 60 to 100 beats per minute (BPM).

In [ ]:
normal_audio, normal_sample  = random_sound(normal_data)
normal_audio

In [ ]:
show_audio_waveform(normal_sample)

In [ ]:
show_audio_spectrum(normal_sample)

In [ ]:
show_spectrogram(normal_sample)

In [ ]:
show_mfcc_features(normal_sample)

### *2. Murmur sound*

Heart murmurs are characterized by audible "whooshing," "roaring," or "rumbling" sounds caused by turbulent blood flow. These sounds typically occur during two specific intervals: between S1 (lub) and S2 (dub), or between S2 and the subsequent S1. Murmurs can serve as clinical indicators for various cardiac disorders, ranging from benign to severe.

In [ ]:
murmur_audio, murmur_sample  = random_sound(murmur_data)
murmur_audio

In [ ]:
show_audio_waveform(murmur_sample)

In [ ]:
show_audio_spectrum(murmur_sample)

In [ ]:
show_spectrogram(murmur_sample)

In [ ]:
show_mfcc_features(murmur_sample)

### *3. Extrasystole sound*

Extrasystoles are occasional, arrhythmic heart sounds characterized by premature or skipped beats. Clinically, these appear as disruptions to the standard cardiac cycle, such as an additional "lub" or "dub" (e.g., "lub-lub dub" or "lub dub-dub").

In [ ]:
extrastole_audio, extrastole_sample  = random_sound(extrastole_data)
extrastole_audio

In [ ]:
show_audio_waveform(extrastole_sample)

In [ ]:
show_audio_spectrum(extrastole_sample)

In [ ]:
show_spectrogram(extrastole_sample)

In [ ]:
show_mfcc_features(extrastole_sample)

### *4. Artifact sound*

The Artifact category encompasses a broad spectrum of external sounds, ranging from feedback squeals and echoes to interference like speech, music, and ambient noise.

In [ ]:
artifact_audio, artifact_sample  = random_sound(artifact_data)
artifact_audio

In [ ]:
show_audio_waveform(artifact_sample)

In [ ]:
show_audio_spectrum(artifact_sample)

In [ ]:
show_spectrogram(artifact_sample)

In [ ]:
show_mfcc_features(artifact_sample)

### *5. Extrahls sound*

Extra heart sounds are identified by an additional acoustic event within the cardiac cycle, deviating from the standard "lub-dub" rhythm to patterns like "lub-lub-dub" or "lub-dub-dub."

In [ ]:
extrahls_audio, extrahls_sample  = random_sound(extrahls_data)
extrahls_audio

In [ ]:
show_audio_waveform(extrahls_sample)

In [ ]:
show_audio_spectrum(extrahls_sample)

In [ ]:
show_spectrogram(extrahls_sample)

In [ ]:
show_mfcc_features(extrahls_sample)

# Data Preprocessing

#### Create imp functions

In [ ]:
def add_noise(data,x):
    noise = np.random.randn(len(data))
    data_noise = data + x * noise
    return data_noise

def shift(data,x):
    return np.roll(data, x)

def stretch(data, rate):
    return librosa.effects.time_stretch(y=data, rate=rate)

def pitch_shift(data, rate, sr=22050):
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=rate)


Processes audio data by extracting MFCC features and applying data augmentation techniques, including noise injection, time-stretching, and time-shifting. A total of 52 features are derived from each sample to serve as model input. <br>
Args: dir_: Path to the directory containing the source audio files.<br>
Returns: data: A list of the 52 features extracted from each file.


In [ ]:
def load_file_data(folder, file_names, duration=10, sr=22050):
    input_length = sr * duration
    features = 52
    data = []

    for file_name in file_names:
        try:
            sound_file = folder + file_name
            X, sr = librosa.load(sound_file, sr=sr, duration=duration)
            dur = librosa.get_duration(y=X, sr=sr)

            # pad audio file to same duration
            if round(dur) < duration:
                print("fixing audio length:", file_name)
                X = librosa.util.fix_length(data=X, size=input_length)

            # original MFCC
            mfccs = np.mean(librosa.feature.mfcc(
                y=X, sr=sr, n_mfcc=features).T, axis=0)
            data.append(mfccs.reshape([-1, 1]))

            # stretch 0.8
            stretch_1 = stretch(X, 0.8)
            mfcc_1 = np.mean(librosa.feature.mfcc(
                y=stretch_1, sr=sr, n_mfcc=features).T, axis=0)
            data.append(mfcc_1.reshape([-1, 1]))

            # stretch 1.2
            stretch_2 = stretch(X, 1.2)
            mfcc_2 = np.mean(librosa.feature.mfcc(
                y=stretch_2, sr=sr, n_mfcc=features).T, axis=0)
            data.append(mfcc_2.reshape([-1, 1]))

        except Exception as e:
            print("Error encountered while parsing file:", file_name, "=>", e)

    return data


## Encoding

In [ ]:
# simple encoding of categories, convert to only 3 types:
# Normal (Include extrahls and extrasystole)
# Murmur
# Artifact

# Map label text to integer
CLASSES = ['artifact','murmur','normal']
NB_CLASSES=len(CLASSES)

# Map integer value to text labels
label_to_int = {k:v for v,k in enumerate(CLASSES)}
print (label_to_int)
print (" ")
int_to_label = {v:k for k,v in label_to_int.items()}
print(int_to_label)

## Data Standardization and Feature Extraction

In [ ]:
# 22 KHz
SAMPLE_RATE = 22050
# 10 seconds
MAX_SOUND_CLIP_DURATION=10

artifact_files = fnmatch.filter(os.listdir(artifact_data), 'artifact*.wav')
artifact_sounds = load_file_data (folder=artifact_data, file_names = artifact_files, duration=MAX_SOUND_CLIP_DURATION)
artifact_labels = [0 for items in artifact_sounds]

normal_files = fnmatch.filter(os.listdir(normal_data), 'normal*.wav')
normal_sounds = load_file_data(folder=normal_data,file_names=normal_files, duration=MAX_SOUND_CLIP_DURATION)
normal_labels = [2 for items in normal_sounds]

extrahls_files = fnmatch.filter(os.listdir(extrahls_data), 'extrahls*.wav')
extrahls_sounds = load_file_data(folder=extrahls_data,file_names=extrahls_files, duration=MAX_SOUND_CLIP_DURATION)
extrahls_labels = [2 for items in extrahls_sounds]

murmur_files = fnmatch.filter(os.listdir(murmur_data), 'murmur*.wav')
murmur_sounds = load_file_data(folder=murmur_data,file_names=murmur_files, duration=MAX_SOUND_CLIP_DURATION)
murmur_labels = [1 for items in murmur_sounds]


extrastole_files = fnmatch.filter(os.listdir(extrastole_data), 'extrastole*.wav')
extrastole_sounds = load_file_data(folder=extrastole_data,file_names=extrastole_files, duration=MAX_SOUND_CLIP_DURATION)
extrastole_labels = [2 for items in extrastole_sounds]

print ("Loading Done")


In [ ]:
print("\nClass-wise sample count (after loading / augmentation if applied):")
print(f"Artifact (label 0):    {len(artifact_labels)}")
print(f"Murmur   (label 1):    {len(murmur_labels)}")
print(f"Normal   (label 2):    {len(normal_labels)}")
print(f"Extrahls (label 2):    {len(extrahls_labels)}")
print(f"Extrastole (label 2):  {len(extrastole_labels)}")

total_samples = (
    len(artifact_labels) +
    len(murmur_labels) +
    len(normal_labels) +
    len(extrahls_labels) +
    len(extrastole_labels)
)

print(f"\nTotal samples: {total_samples}")


In [ ]:
# unlabel_datala files
Bunlabelledtest_files = fnmatch.filter(os.listdir(unlabel_data), 'Bunlabelledtest*.wav')
Bunlabelledtest_sounds = load_file_data(folder=unlabel_data,file_names=Bunlabelledtest_files, duration=MAX_SOUND_CLIP_DURATION)
Bunlabelledtest_labels = [-1 for items in Bunlabelledtest_sounds]

Aunlabelledtest_files = fnmatch.filter(os.listdir(unlabel_data), 'Aunlabelledtest*.wav')
Aunlabelledtest_sounds = load_file_data(folder=unlabel_data,file_names=Aunlabelledtest_files, duration=MAX_SOUND_CLIP_DURATION)
Aunlabelledtest_labels = [-1 for items in Aunlabelledtest_sounds]


print ("Loading of unlabel data done")


In [ ]:
print("\nUnlabelled test data summary:")

print(f"B-unlabelled test samples : {len(Bunlabelledtest_sounds)}")
print(f"A-unlabelled test samples : {len(Aunlabelledtest_sounds)}")

print("\nAssigned labels check (should be -1):")
print(f"B-unlabelled unique labels : {set(Bunlabelledtest_labels)}")
print(f"A-unlabelled unique labels : {set(Aunlabelledtest_labels)}")

print("\nTotal unlabelled test samples:")
print(len(Bunlabelledtest_sounds) + len(Aunlabelledtest_sounds))


## Concatenation

In [ ]:
#combine set-a and set-b
# sound files
x_data = np.concatenate((artifact_sounds, normal_sounds,extrahls_sounds,murmur_sounds,extrastole_sounds))
# contains labels
y_data = np.concatenate((artifact_labels, normal_labels,extrahls_labels,murmur_labels,extrastole_labels))

test_x = np.concatenate((Aunlabelledtest_sounds,Bunlabelledtest_sounds))
test_y = np.concatenate((Aunlabelledtest_labels,Bunlabelledtest_labels))

print ("combined training data record: ",len(y_data), len(test_y))

## Data Split


In [ ]:
# shuffle - whether or not to shuffle the data before splitting. If shuffle=False then stratify must be None.

# split data into Train, Validation and Test
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, train_size=0.8, random_state=42, shuffle=True)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, train_size=0.9, random_state=42, shuffle=True)

# One-Hot encoding for classes
y_train = np.array(tf.keras.utils.to_categorical(y_train, len(CLASSES)))
y_test = np.array(tf.keras.utils.to_categorical(y_test, len(CLASSES)))
y_val = np.array(tf.keras.utils.to_categorical(y_val, len(CLASSES)))
test_y=np.array(tf.keras.utils.to_categorical(test_y, len(CLASSES)))

## Correct Imbalnced Data

In [ ]:
# class weight
TRAIN_IMG_COUNT = 578
COUNT_0 = 40  #artifact
COUNT_1 = 129 #murmur
COUNT_2 = 409 #normal
weight_for_0 = TRAIN_IMG_COUNT / (3 * COUNT_0)
weight_for_1 = TRAIN_IMG_COUNT / (3 * COUNT_1)
weight_for_2 = TRAIN_IMG_COUNT / (3 * COUNT_2)
class_weight = {0: weight_for_0, 1: weight_for_1, 2: weight_for_2}
class_weight

# LSTM Model

In [ ]:
x_train_lstm = x_train
x_val_lstm = x_test
x_test_lstm = x_val

y_train_lstm = y_train
y_val_lstm = y_test
y_test_lstm = y_val

## Build Model

In [ ]:
lstm_model = Sequential([
    tf.keras.layers.Input(shape=(52, 1)),

    # feature extraction
    Conv1D(2048, kernel_size=5, strides=1, padding='same', activation='relu'),
    MaxPooling1D(pool_size=2, strides=2, padding='same'),
    BatchNormalization(),

    Conv1D(1024, kernel_size=5, strides=1, padding='same', activation='relu'),
    MaxPooling1D(pool_size=2, strides=2, padding='same'),
    BatchNormalization(),

    Conv1D(512, kernel_size=5, strides=1, padding='same', activation='relu'),
    MaxPooling1D(pool_size=2, strides=2, padding='same'),
    BatchNormalization(),

    LSTM(256, return_sequences=True),
    LSTM(128),

    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(32, activation='relu'),
    Dropout(0.5),
    # multiclass
    Dense(3, activation='softmax')
])

lstm_model.summary()


In [ ]:
optimiser = tf.keras.optimizers.Adam(learning_rate = 0.0001)

lstm_model.compile(
    optimizer = optimiser,
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)
# callbacks
# 96 , 96 , 96 , 95.99 , 96
cb = [
    EarlyStopping(
        patience = 20,
        monitor = 'val_accuracy',
        mode = 'max',
        restore_best_weights = True
    ),

    ModelCheckpoint(
        "/content/heart_beat_checkpoint/Heart_LSTM_CNN_1.h5",
        save_best_only = True
    )
]

## Train Model

In [ ]:
history = lstm_model.fit(x_train_lstm, y_train_lstm,
                         validation_data=(x_val_lstm, y_val_lstm),
                         batch_size=8, epochs=150,
                         class_weight=class_weight,callbacks = cb )

## Evaluate Model

In [ ]:
lstm_model.evaluate(x_val_lstm, y_val_lstm)

In [ ]:
def plot_loss_curves(history):
  """
  Returns separate loss curves for training and validation metrics.
  Args:
    history: TensorFlow model History object (see: https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/History)
  """
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  accuracy = history.history['accuracy']
  val_accuracy = history.history['val_accuracy']

  epochs = range(len(history.history['loss']))

  # Plot loss
#   plt.plot(epochs, loss, label='training_loss')
#   plt.plot(epochs, val_loss, label='val_loss')
#   plt.title('Loss')
#   plt.xlabel('Epochs')
#   plt.legend()
#   plt.grid()


  # Plot accuracy
  plt.figure()
  plt.grid()
  plt.plot(epochs, accuracy, label='training_accuracy')
  plt.plot(epochs, val_accuracy, label='val_accuracy')
  plt.title('Accuracy')
  plt.xlabel('Epochs')
  plt.legend();

plot_loss_curves(history)

## Results

In [ ]:

classes = ["artifact" ,"murmur ", "normal"]

preds = lstm_model.predict(x_test_lstm)
classpreds = [ np.argmax(t) for t in preds ]
y_testclass = [np.argmax(t) for t in y_test_lstm]
cm = confusion_matrix(y_testclass, classpreds)

plt.figure(figsize=(8, 6), dpi=80, facecolor='w', edgecolor='k')
ax = sns.heatmap(cm, cmap='Blues', annot=True, fmt='d', xticklabels=classes, yticklabels=classes)

plt.title('')
plt.xlabel('Prediction')
plt.ylabel('Truth')
plt.show(ax)

In [ ]:
print(classification_report(y_testclass, classpreds, target_names=classes))

## Prediction

In [ ]:
def heart_prediction(file_path, duration=10, sr=22050):
    classes = ["artifact", "murmur", "normal"]
    input_length = sr * duration

    # Load audio
    X, sr = librosa.load(file_path, sr=sr, duration=duration)
    dur = librosa.get_duration(y=X, sr=sr)

    # Fix length (new librosa API)
    if round(dur) < duration:
        X = librosa.util.fix_length(data=X, size=input_length)

    # Extract MFCC
    mfccs = np.mean(
        librosa.feature.mfcc(
            y=X,
            sr=sr,
            n_mfcc=52,
            n_fft=512,
            hop_length=2048
        ).T,
        axis=0
    )

    mfcc_input = mfccs.reshape(1, 52, 1)

    # Predict
    raw_pred = lstm_model.predict(mfcc_input)

    pred_class = classes[np.argmax(raw_pred)]
    confidence = float(np.max(raw_pred))

    return pred_class, confidence


TEST

In [ ]:
file_path = "/content/artifact__201106110909.wav"
pred, conf = heart_prediction(file_path)
print("Prediction:", pred)
print("Confidence:", conf)

In [ ]:
# Save the trained model in HDF5 (.h5) format
lstm_model.save("lstm_model.h5")
# lstm_model.save("lstm_model.keras")

print("✅ Model saved successfully as lstm_model.keras")
